# === OCR CHAMPION SECTION (merged from final/03_ultimate_pipeline) ===

Один блок для воспроизводимого OCR-пайплайна (local + Kaggle).


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
print('IS_KAGGLE =', IS_KAGGLE)

if IS_KAGGLE:
    req_candidates = [
        Path('/kaggle/working/LentaHack26/final/07_requirements_kaggle.txt'),
        Path('/kaggle/working/LentaHack26b/final/07_requirements_kaggle.txt'),
        Path('/kaggle/working/final/07_requirements_kaggle.txt'),
        Path('final/07_requirements_kaggle.txt'),
        Path('07_requirements_kaggle.txt'),
    ]

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'pip'])
    subprocess.check_call(['apt-get', 'update'])
    subprocess.check_call(['apt-get', 'install', '-y', 'libzbar0'])

    req_file = next((p for p in req_candidates if p.exists()), None)
    if req_file is not None:
        print('Installing from requirements:', req_file)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)])
    else:
        pkgs = [
            'numpy==1.26.4', 'pandas==2.2.3', 'matplotlib==3.8.4', 'tqdm==4.67.1',
            'scipy==1.12.0', 'scikit-image==0.22.0', 'scikit-learn==1.4.2',
            'opencv-python==4.10.0.84', 'opencv-contrib-python==4.10.0.84',
            'pillow==10.4.0', 'pyzbar==0.1.9',
            'paddlepaddle==3.2.0', 'paddleocr==3.2.0', 'gdown==5.2.0', 'kaggle==1.6.17',
            'jupyter==1.1.1', 'ipykernel==6.29.5', 'nbformat==5.10.4',
        ]
        print('Installing inline pinned dependencies...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkgs])

    print('Bootstrap done. Рекомендуется Restart Session и затем Run all.')
else:
    print('Локальный режим: установите зависимости из 08_requirements_local.txt')


In [ ]:
from pathlib import Path
import os


def _unique_paths(paths):
    out = []
    seen = set()
    for p in paths:
        rp = str(Path(p).resolve())
        if rp not in seen:
            seen.add(rp)
            out.append(Path(p).resolve())
    return out


def detect_run_root() -> Path:
    override = os.getenv('RUN_ROOT_OVERRIDE', '').strip()
    if override:
        p = Path(override).expanduser().resolve()
        if p.exists():
            return p

    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / 'final',
        cwd / 'LentaHack26',
        cwd / 'LentaHack26' / 'final',
        cwd / 'LentaHack26b',
        cwd / 'LentaHack26b' / 'final',
    ]

    for parent in [cwd] + list(cwd.parents):
        candidates.extend([
            parent,
            parent / 'final',
            parent / 'LentaHack26',
            parent / 'LentaHack26' / 'final',
            parent / 'LentaHack26b',
            parent / 'LentaHack26b' / 'final',
        ])

    if Path('/kaggle').exists():
        candidates.extend([
            Path('/kaggle/working/LentaHack26'),
            Path('/kaggle/working/LentaHack26/final'),
            Path('/kaggle/working/LentaHack26b'),
            Path('/kaggle/working/LentaHack26b/final'),
            Path('/kaggle/working/final'),
        ])

    candidates = _unique_paths(candidates)

    for c in candidates:
        if (c / 'run_champion_pipeline.py').exists() or (c / 'hypothesis_campaign.py').exists():
            return c

    for c in candidates:
        if (c / 'notebookc9d692d630.ipynb').exists() and (c / 'lenta_tech_life_hack_text.md').exists():
            return c

    return cwd


RUN_ROOT = detect_run_root()
BASE_ROOT = RUN_ROOT.parent if RUN_ROOT.name == 'final' else RUN_ROOT
SEARCH_ROOTS = [RUN_ROOT, BASE_ROOT]


def resolve_path(rel_path: str) -> Path:
    for root in SEARCH_ROOTS:
        p = root / rel_path
        if p.exists():
            return p.resolve()
    return (RUN_ROOT / rel_path).resolve()


ENGINE_NOTEBOOK_NAME = os.getenv('ENGINE_NOTEBOOK_NAME', 'notebookc9d692d630.ipynb').strip() or 'notebookc9d692d630.ipynb'
NOTEBOOK_PATH = resolve_path(ENGINE_NOTEBOOK_NAME)
if not NOTEBOOK_PATH.exists():
    for root in SEARCH_ROOTS:
        candidates = sorted(root.glob('notebook*.ipynb'))
        if candidates:
            NOTEBOOK_PATH = candidates[0].resolve()
            break

DATASET_ROOT = resolve_path('top_crops')
TASK_PATH = resolve_path('lenta_tech_life_hack_text.md')
PRODUCTS_DICT = resolve_path('products_v2_merged.csv')
GOOGLE_DICT = resolve_path('google_dict_normalized.csv')
OUTPUT_ROOT = (RUN_ROOT / 'remote_outputs' / 'final_repro_omega2').resolve()

print('RUN_ROOT =', RUN_ROOT)
print('BASE_ROOT =', BASE_ROOT)
print('DATASET_ROOT =', DATASET_ROOT, 'exists=', DATASET_ROOT.exists())
print('TASK_PATH =', TASK_PATH, 'exists=', TASK_PATH.exists())
print('NOTEBOOK_PATH =', NOTEBOOK_PATH, 'exists=', NOTEBOOK_PATH.exists())
print('PRODUCTS_DICT =', PRODUCTS_DICT, 'exists=', PRODUCTS_DICT.exists())
print('GOOGLE_DICT =', GOOGLE_DICT, 'exists=', GOOGLE_DICT.exists())
print('OUTPUT_ROOT =', OUTPUT_ROOT)


In [ ]:
from pathlib import Path
import os

print("=== PRE-RUN CHECKLIST ===")
print("Kaggle:", Path('/kaggle').exists())
print("RUN_ROOT:", RUN_ROOT)
print("BASE_ROOT:", BASE_ROOT)

required_any_runner = [
    RUN_ROOT / 'run_champion_pipeline.py',
    BASE_ROOT / 'run_champion_pipeline.py',
    RUN_ROOT / 'hypothesis_campaign.py',
    BASE_ROOT / 'hypothesis_campaign.py',
]
runner_found = next((p for p in required_any_runner if p.exists()), None)

checks = {
    'runner_found': runner_found,
    'ocr_engine_notebook': NOTEBOOK_PATH,
    'task_markdown': TASK_PATH,
    'products_dict_csv': PRODUCTS_DICT,
    'dataset_top_crops_dir': DATASET_ROOT,
    'requirements_kaggle': (RUN_ROOT / 'final' / '07_requirements_kaggle.txt') if (RUN_ROOT / 'final').exists() else (BASE_ROOT / 'final' / '07_requirements_kaggle.txt'),
}

ok = True
for name, path in checks.items():
    if path is None:
        exists = False
        path_str = 'None'
    else:
        exists = Path(path).exists()
        path_str = str(path)
    status = 'OK' if exists else 'MISSING'
    print(f"[{status}] {name}: {path_str}")
    ok = ok and exists

if GOOGLE_DICT.exists():
    print(f"[OK] google_dict_normalized.csv: {GOOGLE_DICT}")
else:
    print(f"[WARN] google_dict_normalized.csv not found: {GOOGLE_DICT}")
    print("       Будет попытка скачать через gdown в следующей ячейке.")

if not DATASET_ROOT.exists():
    print("\n[INFO] top_crops не найден локально.")
    print("       Варианты:")
    print("       1) Attach Kaggle Dataset с папкой top_crops")
    print("       2) Attach zip с top_crops")
    print("       3) set os.environ['KAGGLE_DATASET_SLUG']='owner/dataset-name'")

if not ok:
    raise FileNotFoundError("Checklist failed: отсутствуют обязательные файлы/пути.")

print("\nChecklist passed. Можно запускать следующую ячейку.")


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

KAGGLE_DATASET_SLUG = os.getenv('KAGGLE_DATASET_SLUG', '').strip()  # owner/dataset-name


def _extract_top_crops_zip(zip_path: Path, out_root: Path) -> bool:
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_root)
    return DATASET_ROOT.exists()


if not DATASET_ROOT.exists() and Path('/kaggle').exists():
    print('DATASET_ROOT missing, trying to locate in /kaggle/input ...')
    found_dir = None
    for p in Path('/kaggle/input').rglob('top_crops'):
        if p.is_dir():
            found_dir = p
            break

    if found_dir is not None:
        print('Found top_crops directory:', found_dir)
        subprocess.check_call(['cp', '-r', str(found_dir), str(DATASET_ROOT.parent)])
    else:
        zip_candidates = [p for p in Path('/kaggle/input').rglob('*.zip') if 'top_crops' in p.name.lower()]
        if zip_candidates:
            z = sorted(zip_candidates, key=lambda x: x.stat().st_mtime, reverse=True)[0]
            print('Found top_crops zip:', z)
            _extract_top_crops_zip(z, DATASET_ROOT.parent)
        elif KAGGLE_DATASET_SLUG:
            print('Downloading dataset via kaggle CLI:', KAGGLE_DATASET_SLUG)
            subprocess.check_call(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET_SLUG, '-p', str(DATASET_ROOT.parent), '--force'])
            local_zips = [p for p in DATASET_ROOT.parent.glob('*.zip') if 'top_crops' in p.name.lower()]
            if local_zips:
                _extract_top_crops_zip(local_zips[0], DATASET_ROOT.parent)

print('DATASET_ROOT exists =', DATASET_ROOT.exists())


In [ ]:
import subprocess
import sys
import pandas as pd
from pathlib import Path

if not GOOGLE_DICT.exists() and Path('/kaggle').exists():
    file_id = '1xkYv8yRTF-jTMKYxOqKGFcwbTQuBoMYL'
    raw_payload = (RUN_ROOT / 'google_drive_payload').resolve()
    print('Downloading Google dictionary payload...')
    subprocess.check_call([sys.executable, '-m', 'gdown', '--id', file_id, '-O', str(raw_payload)])

    try:
        df = pd.read_csv(raw_payload, sep=';', encoding='cp1251')
        df.to_csv(GOOGLE_DICT, index=False)
    except Exception:
        # Fallback: keep raw file as-is if parsing config differs
        raw_payload.replace(GOOGLE_DICT)
    print('Saved dictionary:', GOOGLE_DICT)
else:
    print('Skip: dictionary already exists or not Kaggle environment.')


In [ ]:
import subprocess
import sys
from pathlib import Path

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

champion_bundle = 'omega2_fixed=data_input/H2,preprocess/H1,ocr/H4,parsers/H2,parsers/H4,qr_barcode/H1,track_merge/H1'

runner = None
runner_mode = None
for p in [RUN_ROOT / 'run_champion_pipeline.py', BASE_ROOT / 'run_champion_pipeline.py']:
    if p.exists():
        runner = p.resolve()
        runner_mode = 'champion_runner'
        break
if runner is None:
    for p in [RUN_ROOT / 'hypothesis_campaign.py', BASE_ROOT / 'hypothesis_campaign.py']:
        if p.exists():
            runner = p.resolve()
            runner_mode = 'hypothesis_campaign'
            break

if runner is None:
    raise FileNotFoundError('Не найден runner: run_champion_pipeline.py или hypothesis_campaign.py')

if runner_mode == 'champion_runner':
    cmd = [
        sys.executable, str(runner),
        '--final-root', str(runner.parent),
        '--output-root', str(OUTPUT_ROOT),
        '--mode', 'sample',
        '--sample-size', '96',
        '--visual-panel-size', '24',
        '--seed', '123',
        '--timeout', '-1',
        '--jupyter-cmd', 'jupyter',
    ]
else:
    cmd = [
        sys.executable, str(runner), 'run_bundle',
        '--project-root', str(runner.parent),
        '--dataset-root', str(DATASET_ROOT),
        '--task-path', str(TASK_PATH),
        '--notebook', str(NOTEBOOK_PATH),
        '--output-root', str(OUTPUT_ROOT),
        '--mode', 'sample',
        '--sample-size', '96',
        '--visual-panel-size', '24',
        '--seed', '123',
        '--timeout', '-1',
        '--jupyter-cmd', 'jupyter',
        '--products-dict-csv', str(PRODUCTS_DICT),
        '--bundle', champion_bundle,
    ]
    if GOOGLE_DICT.exists():
        cmd.extend(['--google-dict-csv', str(GOOGLE_DICT)])

print('Runner mode:', runner_mode)
print('Command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(runner.parent))


In [ ]:
import json
import pandas as pd
from pathlib import Path

table_path = OUTPUT_ROOT / 'candidate_table.csv'

if table_path.exists():
    cand = pd.read_csv(table_path)
    display(cand.tail(5))
    run_id = str(cand.iloc[-1]['run_id'])
    run_dir = OUTPUT_ROOT / run_id
else:
    run_dirs = sorted(
        [d for d in OUTPUT_ROOT.iterdir() if d.is_dir() and (d / 'metrics_v2.json').exists()],
        key=lambda d: d.stat().st_mtime,
    )
    assert run_dirs, f'No run directories with metrics_v2.json under {OUTPUT_ROOT}'
    run_dir = run_dirs[-1]
    run_id = run_dir.name

metrics_path = run_dir / 'metrics_v2.json'
metrics = json.loads(metrics_path.read_text())

print('run_id =', run_id)
print('rows =', metrics.get('rows'))
print('proxy_score =', metrics.get('proxy_score'))
print('case_proxy_v2 =', metrics.get('case_proxy_v2'))
print('price_any_fill =', metrics.get('price_any_fill'))
print('product_name_fill =', (metrics.get('fill_rate', {}) or {}).get('product_name'))
print('barcode_fill =', (metrics.get('fill_rate', {}) or {}).get('barcode'))


In [ ]:
import pandas as pd

result_csv = run_dir / 'outputs_ocr_baseline' / 'result.csv'
out_df = pd.read_csv(result_csv)

cols = ['filename', 'product_name', 'price_default', 'price_discount', 'price_card', 'barcode']
print('result rows =', len(out_df))
display(out_df[cols].sample(min(12, len(out_df)), random_state=42))

print('\nРучная проверка: откройте 10-15 изображений и сверяйте поля с ценниками.')
print('Пути: top_crops/<filename>/...')


# === END OCR CHAMPION SECTION ===
